# 📚 Resum de Funcions – Pràctica 2: Image Filtering & Edge Detection

Aquest notebook recull totes les funcions de biblioteques externes que hem utilitzat al llarg de la pràctica 2 de Visió per Computador.
Inclou exemples bàsics d'ús per a cada funció.

**Biblioteques principals:**
-  (scikit-image)
- 
- 
- 


## 0. Imports generals


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import pyplot as plt


---
## 1.  – Lectura i visualització d'imatges


In [ ]:
from skimage import io

# Llegir una imatge (retorna array numpy)
# img = io.imread("ruta/imatge.jpg")
# - Imatge RGB → shape (H, W, 3), dtype uint8, valors [0, 255]
# - Imatge grayscale → shape (H, W), dtype uint8, valors [0, 255]

img = io.imread("./images_notebook/oryx.jpg")
print("Shape:", img.shape)
print("Dtype:", img.dtype)
print("Min:", img.min(), "Max:", img.max())


---
## 2.  – Conversions de color


In [ ]:
from skimage.color import rgb2gray

# Converteix imatge RGB a escala de grisos
# - Entrada: uint8 [0, 255] RGB
# - Sortida: float64 [0.0, 1.0] (grisos)

img_gray = rgb2gray(img)
print("Shape:", img_gray.shape)  # (H, W) – sense canal de color
print("Dtype:", img_gray.dtype)  # float64
print("Min:", img_gray.min(), "Max:", img_gray.max())


---
## 3.  – Conversions de tipus (, )


In [ ]:
from skimage import img_as_float, img_as_ubyte

# img_as_float: converteix a float64 normalitzat a [0.0, 1.0]
# - Útil per fer operacions aritmètiques sobre la imatge
img_float = img_as_float(img)
print("img_as_float → dtype:", img_float.dtype, "min:", img_float.min(), "max:", img_float.max())

# img_as_ubyte: converteix a uint8 [0, 255]
# - Útil per mostrar o desar imatges
# ⚠️ Atenció: si l'entrada té valors negatius o > 1 pot generar errors/clipping
img_ubyte = img_as_ubyte(img_gray)
print("img_as_ubyte → dtype:", img_ubyte.dtype, "min:", img_ubyte.min(), "max:", img_ubyte.max())


---
## 4.  – Escala i redimensionat


In [ ]:
from skimage.transform import rescale, resize

# rescale: escala la imatge per un factor
# - preserve_range=True → manté el rang de valors originals
# - anti_aliasing=True → aplica suavitzat per evitar aliasing en downsizing
img_small = rescale(img_gray, 1/20, preserve_range=True, anti_aliasing=True)
img_big   = rescale(img_gray, 20,   preserve_range=True, anti_aliasing=True)
print("Original shape:", img_gray.shape)
print("1/20 shape:", img_small.shape)
print("x20 shape:",  img_big.shape)

# resize: canvia la mida a unes dimensions concretes
# - Sempre retorna float64 [0.0, 1.0]
img_resized = resize(img, (256, 256), anti_aliasing=True)
print("Resized shape:", img_resized.shape, "dtype:", img_resized.dtype)


---
## 5.  – Histograma


In [ ]:
from skimage.exposure import histogram
import matplotlib.pyplot as plt
import numpy as np

# ════════════════════════════════════════════
# MANERA 1: skimage.exposure.histogram
# ════════════════════════════════════════════
# Retorna (hist, bin_centers)
# - hist:        freqüències de cada bin
# - bin_centers: valor central de cada bin
# Funciona amb float [0,1] i uint8 [0,255]

# hist, bin_centers = histogram(img_gray, nbins=64)
# plt.bar(bin_centers, hist, width=bin_centers[1] - bin_centers[0])
# plt.title("Histograma – skimage")
# plt.show()

# ════════════════════════════════════════════
# MANERA 2: matplotlib plt.hist
# ════════════════════════════════════════════
# Més ràpid, no cal calcular bins manualment
# img.ravel() → aplana la imatge a 1D

# plt.hist(img_gray.ravel(), bins=64, color='steelblue', edgecolor='none')
# plt.title("Histograma – matplotlib")
# plt.xlabel("Intensitat")
# plt.ylabel("Freqüència")
# plt.show()

# ════════════════════════════════════════════
# COMPARACIÓ VISUAL COSTAT A COSTAT
# ════════════════════════════════════════════
# Generem una imatge de prova
img_demo = np.random.randint(0, 256, (100, 100), dtype=np.uint8)

hist_sk, bin_centers = histogram(img_demo, nbins=32)

fig, axes = plt.subplots(1, 2, figsize=(12, 3))

# skimage
axes[0].bar(bin_centers, hist_sk, width=bin_centers[1] - bin_centers[0],
            color='steelblue', edgecolor='none')
axes[0].set_title("skimage.exposure.histogram + plt.bar")
axes[0].set_xlabel("Intensitat")
axes[0].set_ylabel("Freqüència")

# matplotlib
axes[1].hist(img_demo.ravel(), bins=32, color='tomato', edgecolor='none')
axes[1].set_title("matplotlib plt.hist")
axes[1].set_xlabel("Intensitat")
axes[1].set_ylabel("Freqüència")

plt.tight_layout()
plt.show()

# ⚠️  Diferència clau:
#   - skimage.histogram → retorna arrays, útil si necessites els valors numèrics
#   - plt.hist          → directe, però no retorna arrays útils per operar


---
## 6.  – Convolució manual


In [ ]:
from scipy.ndimage import convolve

# convolve: aplica una convolució 2D amb una màscara personalitzada
# - mode="constant" → considera els píxels fora de la imatge com a 0
# - mode="reflect"  → reflecteix la imatge als marges (per defecte)

# Exemple: filtre de mitjana 5x5
mask = np.ones((5, 5)) / 25.0  # màscara normalitzada
img_conv = convolve(img_gray, mask)
print("Resultat convolució → shape:", img_conv.shape, "dtype:", img_conv.dtype)

# ⚠️ Per aplicar convolució a imatge RGB canal a canal:
# filtered[:,:,0] = convolve(img[:,:,0], mask)
# filtered[:,:,1] = convolve(img[:,:,1], mask)
# filtered[:,:,2] = convolve(img[:,:,2], mask)


---
## 7.  – Filtre Gaussià


In [ ]:
from skimage.filters import gaussian

# gaussian: aplica un filtre gaussià a la imatge
# - sigma: desviació estàndard del kernel → controla la intensitat del suavitzat
# - Retorna float64

img_g1 = gaussian(img_gray, sigma=1.0)
img_g2 = gaussian(img_gray, sigma=2.0)
img_g5 = gaussian(img_gray, sigma=5.0)

fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharex=True, sharey=True)
for ax, im, t in zip(axes, [img_gray, img_g1, img_g2, img_g5], 
                        ["Original", "σ=1", "σ=2", "σ=5"]):
    ax.imshow(im, cmap="gray")
    ax.set_title(t)
    ax.axis("off")
plt.tight_layout()
plt.show()

# Ús en imatges híbrides:
# img_low_freq  = gaussian(img, sigma=4)     # freqüències baixes
# img_high_freq = img - img_low_freq          # freqüències altes


---
## 8.  – Filtre de Mediana


In [ ]:
from skimage.filters import median
from skimage.morphology import disk

# median: aplica el filtre de mediana usant un element estructurant (footprint)
# - disk(r): cercle de radi r → defineix el veïnatge del filtre
# - Molt eficaç per a soroll salt-and-pepper

img_ubyte_gray = img_as_ubyte(img_gray)
img_median = median(img_ubyte_gray, disk(3))
print("Median result → dtype:", img_median.dtype, "shape:", img_median.shape)


---
## 9.  – Elements estructurants


In [ ]:
from skimage.morphology import disk, rectangle, diamond

# disk(r): element circular de radi r
print("disk(2):")
print(disk(2))

# rectangle(r, c): element rectangular r x c
print("
rectangle(3, 7):")
print(rectangle(3, 7))

# diamond(r): element en forma de rombe de radi r
print("
diamond(2):")
print(diamond(2))

# Ús típic: com a footprint del filtre de mediana
# median(image, disk(5))
# median(image, rectangle(5, 20))
# median(image, diamond(3))


---
## 10.  – Soroll sintètic ()


In [ ]:
from skimage.util import random_noise

# random_noise: afegeix soroll a una imatge
# - Retorna sempre float64 [0.0, 1.0]

# Soroll gaussià:
#   mean  → mitja del soroll
#   var   → variància (intensitat del soroll)
# img_noisy_gauss = random_noise(img_gray, mean=0, var=0.01)

# Soroll salt-and-pepper (impulsiu):
#   mode="s&p"   → barreja sal i pebre
#   mode="salt" → només píxels blancs
#   mode="pepper" → només píxels negres
# img_noisy_sp = random_noise(img_gray, mode="s&p")


---
## 11.  – Detecció de contorns (Sobel, Prewitt)


In [ ]:
from skimage.filters import sobel, sobel_h, sobel_v
from skimage.filters import prewitt, prewitt_h, prewitt_v

# Operadors de gradient per a detecció de contorns
# - Retornen float64 amb la magnitud del gradient
# - sobel / prewitt:   gradient total (magnitud)
# - sobel_h / prewitt_h: gradient horitzontal (derivada en X)
# - sobel_v / prewitt_v: gradient vertical   (derivada en Y)

# ⚠️ Funcionen millor amb imatges float, no amb uint8
#    → Convertir amb img_as_float() o rgb2gray() (ja retorna float)

# img_gray_float = img_as_float(img_ubyte_gray)
# edges_sobel   = sobel(img_gray_float)
# edges_prewitt = prewitt(img_gray_float)
# edges_sobel_h = sobel_h(img_gray_float)   # gradient horitzontal
# edges_sobel_v = sobel_v(img_gray_float)   # gradient vertical

# Magnitud combinada manual:
# magnitude = np.sqrt(sobel_h(img)**2 + sobel_v(img)**2)


---
## 12.  – Detector de Canny


In [ ]:
from skimage.feature import canny

# canny: detector de contorns multi-etapa
# - sigma:      suavitzat gaussià previ (elimina soroll)
# - low_threshold / high_threshold: llindars per a hysteresis thresholding
# - Retorna imatge booleana (True = contorn)

# ⚠️ Requereix imatge uint8 (0–255)
#    → Usar img_as_ubyte(rgb2gray(img))

# img_uint8 = img_as_ubyte(rgb2gray(img))
# edges_canny = canny(img_uint8, sigma=1.0, low_threshold=10, high_threshold=20)

# A major sigma → més suavitzat → menys soroll detectat
# A major threshold → menys contorns (només els més forts)


---
## 13.  – Segmentació (Quickshift, SLIC)


In [ ]:
from skimage.segmentation import quickshift, mark_boundaries
import skimage.segmentation as segmentation

# quickshift: segmentació basada en medi shift al color space
# - kernel_size: mida del kernel per estimar densitat
# - max_dist:    distància màxima per unir regions
# - ratio:       balanç entre color i proximitat espacial [0..1]
# segments = quickshift(img_rgb, kernel_size=5, max_dist=20, ratio=0.8)

# segmentation.slic: segmentació en superpíxels (SLIC)
# - n_segments:   nombre aproximat de superpíxels
# - compactness:  balanç entre color i forma (> → més quadrats)
# - sigma:        suavitzat previ
# - start_label=1: els labels comencen per 1
# segments_slic = segmentation.slic(img_rgb, n_segments=200, compactness=25, sigma=1, start_label=1)

# mark_boundaries: dibuixa els contorns de la segmentació sobre la imatge
# - Retorna imatge float [0.0, 1.0]
# img_with_borders = mark_boundaries(img_rgb, segments)

# segmentation.mark_boundaries → equivalent a mark_boundaries si s'importa el mòdul


---
## 14.  – Visualització d'imatges


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Imatge de prova
img_demo = np.random.rand(50, 50)

# ════════════════════════════════════════════
# MANERA 1: fig.add_subplot  (estil antic)
# ════════════════════════════════════════════
# - Crees plt.figure() primer
# - Afegeixes subplots un a un amb add_subplot(rows, cols, index)
# - L'índex és 1-BASED (comença per 1, no 0)
# - Les crides plt.xxx() s'apliquen al subplot "actiu" (el darrer creat)

fig = plt.figure(figsize=(12, 3))

fig.add_subplot(1, 3, 1)          # subplot 1 de 3 (índex 1-based)
plt.imshow(img_demo, cmap='viridis')
plt.title("add_subplot – 1")
plt.axis('off')

fig.add_subplot(1, 3, 2)          # subplot 2 de 3
plt.imshow(img_demo * 0.5, cmap='plasma')
plt.title("add_subplot – 2")
plt.axis('off')

fig.add_subplot(1, 3, 3)          # subplot 3 de 3
plt.imshow(1 - img_demo, cmap='gray')
plt.title("add_subplot – 3")
plt.axis('off')

plt.tight_layout()
plt.show()

# ════════════════════════════════════════════
# MANERA 2: plt.subplots  (estil modern, recomanat ✅)
# ════════════════════════════════════════════
# - Crees figura I tots els subplots alhora
# - L'índex és 0-BASED (comença per 0)
# - Accés explícit a cada subplot via ax[i]
# - Suporta sharex=True / sharey=True per sincronitzar eixos
# - En 2D: ax[fila, columna] o ax[fila][columna]

fig, ax = plt.subplots(1, 3, figsize=(12, 3), sharex=True, sharey=True)

ax[0].imshow(img_demo, cmap='viridis')     # índex 0-based
ax[0].set_title("subplots – ax[0]")
ax[0].axis('off')

ax[1].imshow(img_demo * 0.5, cmap='plasma')
ax[1].set_title("subplots – ax[1]")
ax[1].axis('off')

ax[2].imshow(1 - img_demo, cmap='gray')
ax[2].set_title("subplots – ax[2]")
ax[2].axis('off')

plt.tight_layout()
plt.show()

# ════════════════════════════════════════════
# TAULA COMPARATIVA
# ════════════════════════════════════════════
# │                   │ fig.add_subplot │ plt.subplots + ax[i] │
# │ Indexació         │ 1-based         │ 0-based              │
# │ Creació subplots  │ un a un         │ tots alhora          │
# │ Referència        │ implícita       │ explícita ax[i]      │
# │ sharex / sharey   │ ❌ difícil      │ ✅ directe           │
# │ Subplots 2D       │ ❌ incòmode     │ ✅ ax[i,j]           │
# │ Estil             │ antic           │ modern, recomanat    │

# ⚠️  Nota: plt.title(), plt.imshow()... actuen sobre el subplot "actiu".
#           Amb ax[i].set_title(), ax[i].imshow()... és sempre explícit i segur.


---
## 15.  – Operacions sobre arrays/imatges


In [ ]:
import numpy as np

# === Crear arrays ===
# np.zeros((H, W))           → imatge negra (float)
# np.zeros_like(img)         → array de zeros amb la mateixa shape i dtype
# np.ones((H, W))            → imatge blanca
# np.array([...])            → crear array des d'una llista

# === Estadística ===
# np.sum(arr)                → suma de tots els elements
# np.sqrt(arr)               → arrel quadrada element a element
# np.unique(arr)             → valors únics

# === Tipus de dades ===
# np.float64 / np.uint8      → tipos de dades
# arr.astype(np.uint8)       → convertir dtype manualment

# Exemple típic: màscara de convolució normalitzada
mask = np.ones((5, 5))
mask = mask / np.sum(mask)
print("Màscara 5x5 normalitzada:")
print(mask)

# Exemple: càlcul de la magnitud del gradient
# magnitude = np.sqrt(grad_x**2 + grad_y**2)

# Exemple: zeros_like per crear una imatge filtrada
# filtered = np.zeros_like(img, dtype=np.float64)


---
## 📋 Taula resum de funcions

| Funció | Mòdul | Entrada | Retorna (dtype · shape · rang) | Notes |
|--------|-------|---------|-------------------------------|-------|
| `io.imread(path)` | `skimage.io` | ruta fitxer | `uint8` · `(H,W,3)` RGB / `(H,W)` gray · `[0,255]` | RGBA → `(H,W,4)` |
| `rgb2gray(img)` | `skimage.color` | `(H,W,3)` qualsevol dtype | `float64` · `(H,W)` · `[0.0, 1.0]` | Perd el canal de color |
| `img_as_float(img)` | `skimage` | qualsevol dtype | `float64` · mateixa shape · `[0.0, 1.0]` | uint8÷255 automàtic |
| `img_as_ubyte(img)` | `skimage` | `float64` · `[0.0, 1.0]` | `uint8` · mateixa shape · `[0, 255]` | ⚠️ Error/clipping si valors fora de `[0,1]` |
| `rescale(img, factor, ...)` | `skimage.transform` | qualsevol dtype/shape | `float64` · nova shape · `[0.0, 1.0]` | Normalitza automàticament des de uint8 |
| `resize(img, (H,W), ...)` | `skimage.transform` | qualsevol dtype/shape (RGB o gray) | `float64` · `(H,W)` o `(H,W,3)` · `[0.0, 1.0]` | Normalitza automàticament des de uint8 |
| `histogram(img, nbins)` | `skimage.exposure` | qualsevol | `(hist, bin_centers)` arrays 1D | No modifica la imatge |
| `convolve(img, mask, mode)` | `scipy.ndimage` | qualsevol dtype | mateix dtype que l'entrada · mateixa shape | Si mask és float → resultat float |
| `gaussian(img, sigma)` | `skimage.filters` | qualsevol dtype | `float64` · mateixa shape · ≈`[0.0, 1.0]` | Funciona en RGB i gray |
| `median(img, footprint)` | `skimage.filters` | qualsevol dtype (millor `uint8`) | **mateix dtype que l'entrada** · mateixa shape | ⚠️ Si entres float → surt float |
| `disk(r)` | `skimage.morphology` | radi `r` (int) | `uint8` · `(2r+1, 2r+1)` · valors `{0,1}` | Footprint per a `median` |
| `rectangle(r, c)` | `skimage.morphology` | `r`, `c` (int) | `uint8` · `(r, c)` · valors `{0,1}` | Footprint per a `median` |
| `diamond(r)` | `skimage.morphology` | radi `r` (int) | `uint8` · forma rombe · valors `{0,1}` | Footprint per a `median` |
| `random_noise(img, mode, ...)` | `skimage.util` | qualsevol | `float64` · mateixa shape · `[0.0, 1.0]` | ⚠️ Sempre retorna float64, independentment de l'entrada |
| `sobel(img)` | `skimage.filters` | `float64` recomanat | `float64` · mateixa shape · **pot ser negatiu** | ⚠️ `img_as_ubyte` falla amb valors negatius |
| `sobel_h(img)` | `skimage.filters` | `float64` recomanat | `float64` · mateixa shape · **pot ser negatiu** | Gradient horitzontal (derivada en X) |
| `sobel_v(img)` | `skimage.filters` | `float64` recomanat | `float64` · mateixa shape · **pot ser negatiu** | Gradient vertical (derivada en Y) |
| `prewitt(img)` | `skimage.filters` | `float64` recomanat | `float64` · mateixa shape · **pot ser negatiu** | ⚠️ `img_as_ubyte` falla amb valors negatius |
| `prewitt_h(img)` | `skimage.filters` | `float64` recomanat | `float64` · mateixa shape · **pot ser negatiu** | Gradient horitzontal |
| `prewitt_v(img)` | `skimage.filters` | `float64` recomanat | `float64` · mateixa shape · **pot ser negatiu** | Gradient vertical |
| `canny(img, sigma, ...)` | `skimage.feature` | `uint8` obligatori `[0,255]` | `bool` · mateixa shape · `{True, False}` | ⚠️ Falla amb float; True = contorn |
| `quickshift(img, ...)` | `skimage.segmentation` | RGB `uint8` o `float` | `int32` · `(H,W)` · etiquetes de segment | Cada píxel té l'ID del seu segment |
| `segmentation.slic(img, ...)` | `skimage.segmentation` | RGB qualsevol dtype | `int32` · `(H,W)` · etiquetes de segment | `start_label=1` → labels des de 1 |
| `mark_boundaries(img, seg)` | `skimage.segmentation` | img qualsevol + seg int | `float64` · `(H,W,3)` · `[0.0, 1.0]` | Retorna sempre RGB float |
| `plt.imshow(img, cmap)` | `matplotlib.pyplot` | qualsevol | objecte `AxesImage` (no importa) | `cmap='gray'` per grisos |
| `plt.subplots(r, c, ...)` | `matplotlib.pyplot` | dims i opcions | `(Figure, array of Axes)` | `ax[i]` per subplot i (0-based) |
| `plt.hist(arr, bins)` | `matplotlib.pyplot` | array 1D | `(n, bins, patches)` (rarament útil) | Usar `.ravel()` per aplanar imatge |
| `plt.bar(x, height, width)` | `matplotlib.pyplot` | arrays 1D | objecte `BarContainer` | Per visualitzar `skimage.histogram` |
| `plt.tight_layout()` | `matplotlib.pyplot` | — | `None` | Ajusta espais entre subplots |
| `np.zeros((H, W))` | `numpy` | shape | `float64` · `(H,W)` · tot zeros | Per defecte float64 |
| `np.zeros_like(img)` | `numpy` | array | **mateix dtype i shape** que entrada · tot zeros | Útil per crear imatge filtrada buida |
| `np.ones((H, W))` | `numpy` | shape | `float64` · `(H,W)` · tot uns | Per defecte float64 |
| `np.sum(arr)` | `numpy` | array | escalar (o array si `axis=`) | Per normalitzar màscares |
| `np.sqrt(arr)` | `numpy` | array | `float64` · mateixa shape | Per calcular magnitud del gradient |
| `np.unique(arr)` | `numpy` | array | array 1D ordenat de valors únics | Útil per veure etiquetes de segmentació |
| `arr.astype(dtype)` | `numpy` | array + dtype | nou array amb el dtype indicat | No normalitza (uint8→float: /255 NO fet) |
